# Model 4 — Triplet Metric-Learning Network on Leafsnap (Google Colab)

This notebook trains and evaluates an ImageNet-pretrained **ResNet-50 metric-learning network** on all 185 Leafsnap species. Instead of a softmax classifier, it learns a 256-dimensional L2-normalized embedding using semi-hard triplet loss.

## Development map

1. Configure Colab, GPU, Drive and reproducibility.
2. Copy `archive.zip` from Drive and extract it to fast local Colab storage.
3. Validate the official TSV manifest and use only the 30,866 original RGB images.
4. Load the exact leakage-aware 70/15/15 group split shared by Models 1–3, or create it identically on a first run.
5. Build class-balanced P×K batches so every anchor has positives and negatives.
6. Train a ResNet-50 backbone plus projection head using semi-hard cosine triplets.
7. Select and resume checkpoints using validation nearest-neighbour Recall@1.
8. Build a reference dictionary from training images only.
9. Evaluate closed-set Top-1/Top-5 retrieval, F1, lab/field gap and embedding margins.
10. Retrieve the nearest species and reference images for a single query.

### Dataset assumptions

- Drive archive: `MyDrive/Major Project/archive.zip`
- Expected records: 30,866 (23,147 lab + 7,719 field)
- Expected classes: 185 overall; field observations cover 184 classes
- `dataset/segmented` contains masks, not extra training images
- Lab views from the same likely specimen are kept in one split to reduce leakage
- Reference prototypes are built only from the training split
- Open-set rejection requires separately calibrated thresholds; this notebook reports the required 185-species closed-set experiment

> Run the notebook from top to bottom only after the Drive upload has completed. Training results are intentionally not pre-filled; they must come from the executed run.


## 1. Setup

Use **Runtime → Change runtime type → GPU** before running. `timm` supplies the pretrained ResNet-50 backbone and `pytorch-metric-learning` supplies the tested cosine triplet miner and loss.


In [ ]:
%pip install -q -U timm pytorch-metric-learning


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import hashlib
import json
import os
import random
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from pytorch_metric_learning import distances, losses, miners
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix, f1_score
)
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from tqdm.auto import tqdm

# -------------------------
# User-editable configuration
# -------------------------
DRIVE_PROJECT = Path('/content/drive/MyDrive/Major Project')
ARCHIVE_PATH = DRIVE_PROJECT / 'archive.zip'
LOCAL_ARCHIVE = Path('/content/archive.zip')
EXTRACT_ROOT = Path('/content/leafsnap_data')
SPLIT_PATH = DRIVE_PROJECT / 'splits/leafsnap_group_split_seed42.csv'
SPLIT_META_PATH = DRIVE_PROJECT / 'splits/leafsnap_group_split_seed42.json'
RUN_DIR = DRIVE_PROJECT / 'checkpoints/triplet_resnet50'
RESULTS_DIR = DRIVE_PROJECT / 'results/triplet_resnet50'

SEED = 42
IMAGE_SIZE = 224
BACKBONE_NAME = 'resnet50.a1_in1k'
EMBEDDING_DIM = 256
CLASSES_PER_BATCH = 16      # Reduce to 8 only if Colab reports CUDA out of memory
SAMPLES_PER_CLASS = 4
TRAIN_BATCH_SIZE = CLASSES_PER_BATCH * SAMPLES_PER_CLASS
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 2
EPOCHS = 40
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
TRIPLET_MARGIN = 0.2
REFERENCE_IMAGES_PER_CLASS = 10
EARLY_STOPPING_PATIENCE = 8
RESUME = True
FORCE_RECREATE_SPLIT = False

for directory in [SPLIT_PATH.parent, RUN_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('GPU not detected. Select Runtime > Change runtime type > GPU, then rerun.')

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.titlesize': 13, 'axes.labelsize': 11})

print('Device:', device, '-', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| timm:', timm.__version__, '| scikit-learn:', sklearn.__version__)
print('Drive archive:', ARCHIVE_PATH)


## 2. Stage and validate the dataset

The archive remains permanent in Drive. It is copied and extracted under `/content` because training directly from thousands of Drive files is slow.

In [ ]:
assert ARCHIVE_PATH.exists(), f'Archive not found: {ARCHIVE_PATH}. Wait for the Drive upload to finish.'

drive_size = ARCHIVE_PATH.stat().st_size
if not LOCAL_ARCHIVE.exists() or LOCAL_ARCHIVE.stat().st_size != drive_size:
    print(f'Copying {drive_size / 2**30:.2f} GiB from Drive to Colab...')
    shutil.copy2(ARCHIVE_PATH, LOCAL_ARCHIVE)
else:
    print('Local archive already matches the Drive archive.')

extract_marker = EXTRACT_ROOT / '.extraction_complete'
if not extract_marker.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting archive...')
    shutil.unpack_archive(LOCAL_ARCHIVE, EXTRACT_ROOT)
    extract_marker.write_text('complete', encoding='utf-8')
else:
    print('Existing extraction detected.')

manifest_candidates = list(EXTRACT_ROOT.rglob('leafsnap-dataset-images.txt'))
assert len(manifest_candidates) == 1, (
    f'Expected exactly one manifest, found {len(manifest_candidates)}: {manifest_candidates}'
)
MANIFEST_PATH = manifest_candidates[0]

def locate_dataset_root(manifest_path: Path) -> Path:
    candidates = [manifest_path.parent / 'leafsnap-dataset', manifest_path.parent, *manifest_path.parents]
    for candidate in candidates:
        if (candidate / 'dataset/images/lab').exists() and (candidate / 'dataset/images/field').exists():
            return candidate
    raise FileNotFoundError('Could not locate leafsnap-dataset/dataset/images after extraction.')

DATASET_ROOT = locate_dataset_root(MANIFEST_PATH)
IMAGES_ROOT = DATASET_ROOT / 'dataset/images'
MASKS_ROOT = DATASET_ROOT / 'dataset/segmented'
manifest_sha256 = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
metadata = pd.read_csv(MANIFEST_PATH, sep='\t')

required_columns = {'file_id', 'image_path', 'segmented_path', 'species', 'source'}
assert required_columns.issubset(metadata.columns), f'Missing columns: {required_columns - set(metadata.columns)}'
assert len(metadata) == 30_866, f'Expected 30,866 records, found {len(metadata):,}'
assert metadata['file_id'].is_unique and metadata['image_path'].is_unique
assert metadata['species'].nunique() == 185
assert metadata['source'].value_counts().to_dict() == {'lab': 23_147, 'field': 7_719}

missing_originals = [p for p in metadata['image_path'] if not (DATASET_ROOT / p).is_file()]
assert not missing_originals, f'Missing original images, first examples: {missing_originals[:5]}'

print('Manifest:', MANIFEST_PATH)
print('Dataset root:', DATASET_ROOT)
print(f'Validated {len(metadata):,} original images across {metadata.species.nunique()} species.')
print(metadata['source'].value_counts().rename('images'))

## 3. Create or reuse the shared split

The split is created **within each species** so every class appears in train, validation and test. Lab filenames commonly end in four view numbers; those views are grouped. Obvious field sequences are also grouped conservatively. The saved CSV and manifest fingerprint make the exact partition reusable by Models 2–4.

In [ ]:
def derive_group_id(row: pd.Series) -> str:
    stem = Path(row['image_path']).stem
    if row['source'] == 'lab':
        group_stem = re.sub(r'-\d+$', '', stem)
    elif re.fullmatch(r'.+_\d+', stem):
        group_stem = re.sub(r'_\d+$', '', stem)
    elif re.fullmatch(r'\d{14}', stem):
        group_stem = stem[:10]
    else:
        group_stem = stem
    return f"{row['source']}/{row['species']}/{group_stem}"

def build_group_split(frame: pd.DataFrame, seed: int) -> pd.DataFrame:
    split_frame = frame.copy()
    split_frame['group_id'] = split_frame.apply(derive_group_id, axis=1)
    split_frame['split'] = ''
    classes = sorted(split_frame['species'].unique())

    for class_index, species in enumerate(classes):
        species_mask = split_frame['species'].eq(species)
        groups = split_frame.loc[species_mask, 'group_id'].drop_duplicates().to_numpy()
        if len(groups) < 3:
            raise ValueError(f'{species} has only {len(groups)} groups; at least 3 are required.')

        rng = np.random.default_rng(seed + class_index)
        groups = rng.permutation(groups)
        n_train = max(1, int(round(0.70 * len(groups))))
        n_val = max(1, int(round(0.15 * len(groups))))
        if n_train + n_val >= len(groups):
            n_train, n_val = len(groups) - 2, 1

        assignment = {group: 'train' for group in groups[:n_train]}
        assignment.update({group: 'val' for group in groups[n_train:n_train + n_val]})
        assignment.update({group: 'test' for group in groups[n_train + n_val:]})
        split_frame.loc[species_mask, 'split'] = split_frame.loc[species_mask, 'group_id'].map(assignment)

    class_to_index = {name: index for index, name in enumerate(classes)}
    split_frame['class_index'] = split_frame['species'].map(class_to_index).astype(int)
    return split_frame

if SPLIT_PATH.exists() and SPLIT_META_PATH.exists() and not FORCE_RECREATE_SPLIT:
    split_metadata = json.loads(SPLIT_META_PATH.read_text(encoding='utf-8'))
    if split_metadata.get('manifest_sha256') != manifest_sha256:
        raise RuntimeError('Saved split belongs to a different manifest. Set FORCE_RECREATE_SPLIT=True after review.')
    if split_metadata.get('seed') != SEED:
        raise RuntimeError('Saved split uses a different seed. Set FORCE_RECREATE_SPLIT=True after review.')
    split_df = pd.read_csv(SPLIT_PATH)
    print('Loaded existing shared split:', SPLIT_PATH)
else:
    split_df = build_group_split(metadata, SEED)
    split_df.to_csv(SPLIT_PATH, index=False)
    split_metadata = {
        'seed': SEED, 'ratios_by_group': {'train': 0.70, 'val': 0.15, 'test': 0.15},
        'manifest_sha256': manifest_sha256, 'records': int(len(split_df)),
        'classes': int(split_df['species'].nunique()),
        'grouping': 'lab strip final -view; field strip _sequence or 14-digit timestamp suffix'
    }
    SPLIT_META_PATH.write_text(json.dumps(split_metadata, indent=2), encoding='utf-8')
    print('Created shared split:', SPLIT_PATH)

# Split integrity checks
assert len(split_df) == len(metadata)
assert set(split_df['file_id']) == set(metadata['file_id'])
assert split_df['split'].isin(['train', 'val', 'test']).all()
assert split_df.groupby('group_id')['split'].nunique().max() == 1, 'Group leakage detected.'
for split_name in ['train', 'val', 'test']:
    assert split_df.loc[split_df['split'].eq(split_name), 'species'].nunique() == 185

split_summary = pd.crosstab(split_df['split'], split_df['source']).reindex(['train', 'val', 'test'])
split_summary['total'] = split_summary.sum(axis=1)
split_summary['share'] = split_summary['total'] / len(split_df)
display(split_summary)

ax = split_summary[['lab', 'field']].plot(
    kind='bar', stacked=True, color=['#35618f', '#d89b2b'], edgecolor='#333333'
)
ax.set_title('Leafsnap images by split and source')
ax.set_xlabel('Split')
ax.set_ylabel('Images')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Source')
plt.tight_layout()
plt.show()

## 4. Class-balanced metric-learning data pipeline

The image augmentation and ImageNet normalization match Model 1's ResNet-50 pipeline. Each training batch contains `CLASSES_PER_BATCH` species and `SAMPLES_PER_CLASS` images per species. This guarantees positive and negative candidates for the triplet miner and avoids the empty-positive problem caused by ordinary random batches.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.25),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.90, 1.10), fill=255),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

CLASS_NAMES = sorted(split_df['species'].unique())
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

class LeafsnapDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, dataset_root: Path, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.dataset_root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = self.dataset_root / row['image_path']
        with Image.open(image_path) as image_file:
            image = image_file.convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['class_index']), index

class PKBatchSampler(Sampler):
    """Deterministic P-species × K-images batches with uniform class sampling."""
    def __init__(
        self, labels, classes_per_batch, samples_per_class,
        batches_per_epoch, seed
    ):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.classes = np.unique(self.labels)
        self.classes_per_batch = int(classes_per_batch)
        self.samples_per_class = int(samples_per_class)
        self.batches_per_epoch = int(batches_per_epoch)
        self.seed = int(seed)
        self.epoch = 0
        if self.classes_per_batch > len(self.classes):
            raise ValueError('classes_per_batch cannot exceed the number of classes.')
        self.class_to_indices = {
            int(class_index): np.flatnonzero(self.labels == class_index)
            for class_index in self.classes
        }

    def __len__(self):
        return self.batches_per_epoch

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        for _ in range(self.batches_per_epoch):
            selected_classes = rng.choice(
                self.classes, size=self.classes_per_batch, replace=False
            )
            batch_indices = []
            for class_index in selected_classes:
                candidates = self.class_to_indices[int(class_index)]
                selected_indices = rng.choice(
                    candidates,
                    size=self.samples_per_class,
                    replace=len(candidates) < self.samples_per_class,
                )
                batch_indices.extend(selected_indices.tolist())
            rng.shuffle(batch_indices)
            yield batch_indices

train_df = split_df[split_df['split'].eq('train')].reset_index(drop=True)
val_df = split_df[split_df['split'].eq('val')].reset_index(drop=True)
test_df = split_df[split_df['split'].eq('test')].reset_index(drop=True)
train_dataset = LeafsnapDataset(train_df, DATASET_ROOT, train_transform)
val_dataset = LeafsnapDataset(val_df, DATASET_ROOT, eval_transform)
test_dataset = LeafsnapDataset(test_df, DATASET_ROOT, eval_transform)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)
batches_per_epoch = int(np.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))
train_batch_sampler = PKBatchSampler(
    labels=train_df['class_index'].to_numpy(),
    classes_per_batch=CLASSES_PER_BATCH,
    samples_per_class=SAMPLES_PER_CLASS,
    batches_per_epoch=batches_per_epoch,
    seed=SEED,
)
train_loader = DataLoader(
    train_dataset, batch_sampler=train_batch_sampler, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=False, worker_init_fn=seed_worker,
    generator=loader_generator
)
eval_loader_kwargs = dict(
    batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=False, worker_init_fn=seed_worker
)
val_loader = DataLoader(val_dataset, **eval_loader_kwargs)
test_loader = DataLoader(test_dataset, **eval_loader_kwargs)

print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')
print(
    f'Training batches: {batches_per_epoch} × {CLASSES_PER_BATCH} species '
    f'× {SAMPLES_PER_CLASS} images = batch size {TRAIN_BATCH_SIZE}'
)
images, labels, row_indices = next(iter(train_loader))
unique_labels, label_counts = labels.unique(return_counts=True)
assert len(unique_labels) == CLASSES_PER_BATCH
assert label_counts.min().item() == SAMPLES_PER_CLASS
print('Verified P×K batch:', tuple(images.shape), '| species:', len(unique_labels))


In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for axis, image_tensor, label in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(denormalize(image_tensor).permute(1, 2, 0))
    axis.set_title(CLASS_NAMES[int(label)], fontsize=9)
    axis.axis('off')
fig.suptitle('Examples from one class-balanced training batch', y=1.01)
plt.tight_layout()
plt.show()


## 5. Embedding model and semi-hard triplet training utilities

The ImageNet-pretrained ResNet-50 produces features for a two-layer projection head. The final 256-dimensional embedding is L2-normalized, so a dot product is cosine similarity. Semi-hard mining selects negatives that are farther than the positive but still violate the `0.2` margin.

Validation uses leave-one-out nearest-neighbour retrieval inside the validation split. Each query is excluded from its gallery, so it cannot retrieve itself. This validation score selects checkpoints; the held-out test split remains untouched until final evaluation.


In [ ]:
class EmbeddingNet(nn.Module):
    """Pretrained backbone plus projection head producing L2-normalized embeddings."""
    def __init__(self, backbone_name, embedding_dim):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0
        )
        feature_dim = self.backbone.num_features
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, images):
        features = self.backbone(images)
        embeddings = self.projection(features)
        return F.normalize(embeddings, p=2, dim=1)

model = EmbeddingNet(BACKBONE_NAME, EMBEDDING_DIM).to(device)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f'Embedding network: {total_parameters:,} total; {trainable_parameters:,} trainable')

distance = distances.CosineSimilarity()
triplet_miner = miners.TripletMarginMiner(
    margin=TRIPLET_MARGIN, distance=distance, type_of_triplets='semihard'
)
triplet_loss = losses.TripletMarginLoss(
    margin=TRIPLET_MARGIN, distance=distance
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)
USE_AMP = device.type == 'cuda'
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

def atomic_torch_save(payload, destination: Path) -> None:
    """Write locally first, then replace the Drive file to reduce partial-checkpoint risk."""
    local_temporary = Path('/content') / f'.{destination.name}.local.tmp'
    drive_temporary = destination.with_suffix(destination.suffix + '.tmp')
    torch.save(payload, local_temporary)
    shutil.copy2(local_temporary, drive_temporary)
    os.replace(drive_temporary, destination)
    local_temporary.unlink(missing_ok=True)

def capture_rng_state():
    return {
        'python': random.getstate(), 'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(), 'cuda': torch.cuda.get_rng_state_all(),
        'loader_generator': loader_generator.get_state(),
    }

def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])
    loader_generator.set_state(state['loader_generator'])

def run_training_epoch(model, loader, epoch):
    model.train()
    train_batch_sampler.set_epoch(epoch)
    running_loss = mined_triplets = active_batches = 0
    progress = tqdm(loader, desc='train/triplet', leave=False)

    for batch_images, batch_targets, _ in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            embeddings = model(batch_images)
        loss_embeddings = embeddings.float()
        mined_indices = triplet_miner(loss_embeddings, batch_targets)
        batch_triplets = int(mined_indices[0].numel())

        if batch_triplets > 0:
            loss = triplet_loss(loss_embeddings, batch_targets, mined_indices)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            active_batches += 1

        mined_triplets += batch_triplets
        progress.set_postfix(
            loss=f'{running_loss / max(active_batches, 1):.4f}',
            triplets=batch_triplets
        )

    return {
        'loss': running_loss / max(active_batches, 1),
        'mined_triplets_per_batch': mined_triplets / len(loader),
        'active_batch_fraction': active_batches / len(loader),
    }

@torch.inference_mode()
def collect_embeddings(model, loader, description):
    model.eval()
    embedding_batches, label_batches, index_batches = [], [], []
    for batch_images, batch_labels, batch_indices in tqdm(loader, desc=description, leave=False):
        batch_images = batch_images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            batch_embeddings = model(batch_images)
        embedding_batches.append(batch_embeddings.float().cpu())
        label_batches.append(batch_labels.long().cpu())
        index_batches.append(batch_indices.long().cpu())
    return (
        torch.cat(embedding_batches),
        torch.cat(label_batches),
        torch.cat(index_batches),
    )

@torch.inference_mode()
def leave_one_out_recall(embeddings, labels, chunk_size=512):
    """Recall@1/5 with each validation query removed from its own gallery."""
    gallery_embeddings = F.normalize(embeddings, dim=1).to(device)
    gallery_labels = labels.to(device)
    correct_at_1 = correct_at_5 = total = 0

    for start in range(0, len(gallery_embeddings), chunk_size):
        end = min(start + chunk_size, len(gallery_embeddings))
        query_embeddings = gallery_embeddings[start:end]
        query_labels = gallery_labels[start:end]
        similarities = query_embeddings @ gallery_embeddings.T
        local_rows = torch.arange(end - start, device=device)
        global_rows = torch.arange(start, end, device=device)
        similarities[local_rows, global_rows] = -torch.inf
        top_labels = gallery_labels[similarities.topk(5, dim=1).indices]
        matches = top_labels.eq(query_labels[:, None])
        correct_at_1 += matches[:, :1].any(dim=1).sum().item()
        correct_at_5 += matches[:, :5].any(dim=1).sum().item()
        total += end - start

    return {'recall_at_1': correct_at_1 / total, 'recall_at_5': correct_at_5 / total}


## 6. Train with checkpointing and resume

`last.pt` stores the complete training state after each epoch. `best.pt` stores the embedding model with the highest leave-one-out validation Recall@1. Checkpoints include optimizer, scheduler, mixed-precision scaler, sampler epoch and random-number states for reproducible continuation.


In [ ]:
LAST_CHECKPOINT = RUN_DIR / 'last.pt'
BEST_CHECKPOINT = RUN_DIR / 'best.pt'
HISTORY_PATH = RESULTS_DIR / 'training_history.csv'

run_config = {
    'model_family': 'triplet_metric_learning',
    'backbone': BACKBONE_NAME, 'pretrained': True,
    'embedding_dim': EMBEDDING_DIM, 'image_size': IMAGE_SIZE,
    'classes_per_batch': CLASSES_PER_BATCH,
    'samples_per_class': SAMPLES_PER_CLASS,
    'train_batch_size': TRAIN_BATCH_SIZE, 'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'triplet_margin': TRIPLET_MARGIN,
    'miner': 'semi-hard cosine', 'reference_images_per_class': REFERENCE_IMAGES_PER_CLASS,
    'seed': SEED, 'manifest_sha256': manifest_sha256, 'split_path': str(SPLIT_PATH)
}
(RESULTS_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

start_epoch, best_val_recall1, epochs_without_improvement = 0, -1.0, 0
history = []
if RESUME and LAST_CHECKPOINT.exists():
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    if checkpoint['manifest_sha256'] != manifest_sha256 or checkpoint['class_names'] != CLASS_NAMES:
        raise RuntimeError('Checkpoint dataset/classes do not match this run.')
    saved_config = checkpoint.get('config', {})
    if (
        saved_config.get('backbone') != BACKBONE_NAME
        or saved_config.get('embedding_dim') != EMBEDDING_DIM
    ):
        raise RuntimeError('Checkpoint backbone/embedding configuration does not match.')
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_recall1 = checkpoint['best_val_recall1']
    epochs_without_improvement = checkpoint['epochs_without_improvement']
    history = checkpoint['history']
    if 'rng_state' in checkpoint:
        restore_rng_state(checkpoint['rng_state'])
    print(
        f'Resuming from epoch {start_epoch + 1}; '
        f'best validation Recall@1={best_val_recall1:.4f}'
    )

training_started = time.time()
for epoch in range(start_epoch, EPOCHS):
    epoch_started = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    train_metrics = run_training_epoch(model, train_loader, epoch)
    val_embeddings, val_labels, _ = collect_embeddings(model, val_loader, 'validation embeddings')
    val_metrics = leave_one_out_recall(val_embeddings, val_labels)
    scheduler.step()

    epoch_record = {
        'epoch': epoch + 1, 'lr': current_lr,
        'train_triplet_loss': train_metrics['loss'],
        'mined_triplets_per_batch': train_metrics['mined_triplets_per_batch'],
        'active_batch_fraction': train_metrics['active_batch_fraction'],
        'val_recall_at_1': val_metrics['recall_at_1'],
        'val_recall_at_5': val_metrics['recall_at_5'],
        'minutes': (time.time() - epoch_started) / 60,
    }
    history.append(epoch_record)

    improved = val_metrics['recall_at_1'] > best_val_recall1
    if improved:
        best_val_recall1 = val_metrics['recall_at_1']
        epochs_without_improvement = 0
        atomic_torch_save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'class_names': CLASS_NAMES, 'manifest_sha256': manifest_sha256,
            'config': run_config, 'val_metrics': val_metrics
        }, BEST_CHECKPOINT)
    else:
        epochs_without_improvement += 1

    atomic_torch_save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(), 'best_val_recall1': best_val_recall1,
        'epochs_without_improvement': epochs_without_improvement, 'history': history,
        'class_names': CLASS_NAMES, 'manifest_sha256': manifest_sha256,
        'config': run_config, 'rng_state': capture_rng_state()
    }, LAST_CHECKPOINT)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"triplet loss {train_metrics['loss']:.4f} | "
        f"mined/batch {train_metrics['mined_triplets_per_batch']:.1f} | "
        f"active batches {train_metrics['active_batch_fraction']:.1%} | "
        f"val R@1 {val_metrics['recall_at_1']:.4f}, R@5 {val_metrics['recall_at_5']:.4f}"
    )
    del val_embeddings, val_labels
    gc.collect()
    torch.cuda.empty_cache()

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping after {EARLY_STOPPING_PATIENCE} epochs without Recall@1 improvement.')
        break

print(f'Training cell completed in {(time.time() - training_started) / 60:.1f} minutes.')
print('Best checkpoint:', BEST_CHECKPOINT)


In [ ]:
history_df = pd.DataFrame(history)
assert not history_df.empty, 'No training history found.'
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(
    history_df['epoch'], history_df['train_triplet_loss'],
    marker='o', color='#35618f'
)
axes[0].set(title='Semi-hard triplet loss', xlabel='Epoch', ylabel='Loss')
axes[1].plot(
    history_df['epoch'], history_df['val_recall_at_1'],
    marker='o', label='Validation R@1', color='#d89b2b'
)
axes[1].plot(
    history_df['epoch'], history_df['val_recall_at_5'],
    marker='o', label='Validation R@5', color='#8a5a9e'
)
axes[1].set(title='Leave-one-out retrieval', xlabel='Epoch', ylabel='Recall', ylim=(0, 1))
axes[1].legend()
triplet_axis = axes[2]
fraction_axis = triplet_axis.twinx()
triplet_line = triplet_axis.plot(
    history_df['epoch'], history_df['mined_triplets_per_batch'],
    marker='o', label='Mined triplets/batch', color='#2a8c82'
)
fraction_line = fraction_axis.plot(
    history_df['epoch'], history_df['active_batch_fraction'],
    marker='o', label='Active batch fraction', color='#a65045'
)
triplet_axis.set(title='Mining activity', xlabel='Epoch', ylabel='Triplets per batch')
fraction_axis.set(ylabel='Active batch fraction', ylim=(0, 1.05))
mining_lines = triplet_line + fraction_line
triplet_axis.legend(mining_lines, [line.get_label() for line in mining_lines])
plt.tight_layout()
figure_path = RESULTS_DIR / 'training_curves.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
display(history_df.tail(10))
print('Saved:', figure_path)


## 7. Build the reference dictionary and evaluate the test split

The best checkpoint is restored first. Up to ten deterministic training images per species form the reference gallery, and their normalized embeddings are averaged into one normalized species prototype. Test images are classified by cosine similarity to these 185 prototypes.

This is a closed-set evaluation comparable to Models 1–3. Test images never contribute to the dictionary, checkpoint selection or threshold tuning.


In [ ]:
assert BEST_CHECKPOINT.exists(), 'Best checkpoint not found; run the training cell first.'
best_checkpoint = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state'])
model.eval()

reference_parts = []
for class_index, class_frame in train_df.groupby('class_index', sort=True):
    sample_count = min(REFERENCE_IMAGES_PER_CLASS, len(class_frame))
    reference_parts.append(
        class_frame.sample(n=sample_count, random_state=SEED + int(class_index))
    )
reference_df = (
    pd.concat(reference_parts, ignore_index=True)
    .sort_values(['class_index', 'file_id'])
    .reset_index(drop=True)
)
reference_dataset = LeafsnapDataset(reference_df, DATASET_ROOT, eval_transform)
reference_loader = DataLoader(reference_dataset, **eval_loader_kwargs)
reference_embeddings, reference_labels, reference_indices = collect_embeddings(
    model, reference_loader, 'reference embeddings'
)
reference_embeddings = F.normalize(reference_embeddings, dim=1)

prototype_embeddings = []
for class_index in range(NUM_CLASSES):
    class_embeddings = reference_embeddings[reference_labels.eq(class_index)]
    assert len(class_embeddings) > 0, f'No reference embeddings for class {class_index}'
    prototype_embeddings.append(F.normalize(class_embeddings.mean(dim=0), dim=0))
prototype_embeddings = torch.stack(prototype_embeddings)

reference_df.to_csv(RESULTS_DIR / 'reference_gallery.csv', index=False)
torch.save({
    'prototype_embeddings': prototype_embeddings,
    'reference_embeddings': reference_embeddings,
    'reference_labels': reference_labels,
    'class_names': CLASS_NAMES,
    'backbone': BACKBONE_NAME,
    'embedding_dim': EMBEDDING_DIM,
}, RESULTS_DIR / 'reference_dictionary.pt')

test_embeddings, test_labels, test_indices = collect_embeddings(
    model, test_loader, 'test embeddings'
)
test_embeddings = F.normalize(test_embeddings, dim=1)
similarities = test_embeddings @ prototype_embeddings.T
top_similarities, top_indices = similarities.topk(5, dim=1)

row_numbers = torch.arange(len(test_labels))
true_similarities = similarities[row_numbers, test_labels]
incorrect_similarities = similarities.clone()
incorrect_similarities[row_numbers, test_labels] = -torch.inf
best_incorrect_similarities = incorrect_similarities.max(dim=1).values
embedding_margins = true_similarities - best_incorrect_similarities

prediction_rows = []
for position in range(len(test_df)):
    row = test_df.iloc[int(test_indices[position])]
    true_index = int(test_labels[position])
    predicted_indices = top_indices[position].tolist()
    predicted_similarities = top_similarities[position].tolist()
    prediction_rows.append({
        'file_id': row['file_id'], 'image_path': row['image_path'],
        'source': row['source'], 'true_species': CLASS_NAMES[true_index],
        'predicted_species': CLASS_NAMES[predicted_indices[0]],
        'correct_top1': int(predicted_indices[0] == true_index),
        'correct_top5': int(true_index in predicted_indices),
        'top5_species': '|'.join(CLASS_NAMES[index] for index in predicted_indices),
        'top5_cosine_similarity': '|'.join(f'{score:.6f}' for score in predicted_similarities),
        'true_prototype_similarity': float(true_similarities[position]),
        'best_wrong_similarity': float(best_incorrect_similarities[position]),
        'embedding_margin': float(embedding_margins[position]),
        'true_index': true_index, 'predicted_index': predicted_indices[0],
    })
predictions_df = pd.DataFrame(prediction_rows)
predictions_path = RESULTS_DIR / 'test_predictions.csv'
predictions_df.to_csv(predictions_path, index=False)
torch.save(
    {'embeddings': test_embeddings, 'labels': test_labels, 'indices': test_indices},
    RESULTS_DIR / 'test_embeddings.pt'
)

def calculate_metrics(frame):
    y_true = frame['true_index'].to_numpy()
    y_pred = frame['predicted_index'].to_numpy()
    return {
        'images': int(len(frame)), 'observed_classes': int(frame['true_species'].nunique()),
        'top1_accuracy': float(frame['correct_top1'].mean()),
        'top5_accuracy': float(frame['correct_top5'].mean()),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'mean_embedding_margin': float(frame['embedding_margin'].mean()),
    }

metrics = {'overall': calculate_metrics(predictions_df)}
for source_name in ['lab', 'field']:
    metrics[source_name] = calculate_metrics(
        predictions_df[predictions_df['source'].eq(source_name)]
    )
metrics_path = RESULTS_DIR / 'test_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
domain_gap = {
    'top1_accuracy_lab_minus_field':
        metrics['lab']['top1_accuracy'] - metrics['field']['top1_accuracy'],
    'macro_f1_lab_minus_field':
        metrics['lab']['macro_f1'] - metrics['field']['macro_f1'],
}
domain_gap_path = RESULTS_DIR / 'domain_gap.json'
domain_gap_path.write_text(json.dumps(domain_gap, indent=2), encoding='utf-8')

metrics_table = pd.DataFrame(metrics).T
display(metrics_table.style.format({
    'top1_accuracy': '{:.4f}', 'top5_accuracy': '{:.4f}', 'macro_f1': '{:.4f}',
    'weighted_f1': '{:.4f}', 'balanced_accuracy': '{:.4f}',
    'mean_embedding_margin': '{:.4f}',
}))
print('Reference images:', len(reference_df), '| prototypes:', len(prototype_embeddings))
print('Saved predictions:', predictions_path)
print('Saved metrics:', metrics_path)
print('Lab minus field gap:', domain_gap)


In [ ]:
y_true = predictions_df['true_index'].to_numpy()
y_pred = predictions_df['predicted_index'].to_numpy()
report = classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
per_class_df = pd.DataFrame(report).T.loc[CLASS_NAMES].reset_index(names='species')
per_class_path = RESULTS_DIR / 'per_class_metrics.csv'
per_class_df.to_csv(per_class_path, index=False)

confusion = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
flat_order = np.argsort(off_diagonal.ravel())[::-1]
confusion_rows = []
for flat_index in flat_order:
    true_index, predicted_index = np.unravel_index(flat_index, off_diagonal.shape)
    count = int(off_diagonal[true_index, predicted_index])
    if count == 0 or len(confusion_rows) == 20:
        break
    confusion_rows.append({
        'pair': f'{CLASS_NAMES[true_index]} → {CLASS_NAMES[predicted_index]}',
        'images': count,
    })
confusion_pairs_df = pd.DataFrame(confusion_rows)
confusion_pairs_df.to_csv(RESULTS_DIR / 'top_confusion_pairs.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
metric_names = ['top1_accuracy', 'top5_accuracy', 'macro_f1', 'weighted_f1']
metric_labels = ['Top-1', 'Top-5', 'Macro F1', 'Weighted F1']
axes[0, 0].bar(
    metric_labels, [metrics['overall'][name] for name in metric_names],
    color='#35618f'
)
axes[0, 0].set(title='Prototype retrieval metrics', ylabel='Score', ylim=(0, 1))
axes[0, 0].tick_params(axis='x', rotation=25)

worst_classes = per_class_df.nsmallest(20, 'f1-score').sort_values('f1-score')
axes[0, 1].barh(worst_classes['species'], worst_classes['f1-score'], color='#d89b2b')
axes[0, 1].set(title='Lowest 20 per-class F1 scores', xlabel='F1 score', xlim=(0, 1))

if not confusion_pairs_df.empty:
    plot_pairs = confusion_pairs_df.sort_values('images')
    axes[1, 0].barh(plot_pairs['pair'], plot_pairs['images'], color='#8a5a9e')
axes[1, 0].set(title='Most frequent prototype confusions', xlabel='Test images')

correct_margins = predictions_df.loc[predictions_df['correct_top1'].eq(1), 'embedding_margin']
incorrect_margins = predictions_df.loc[predictions_df['correct_top1'].eq(0), 'embedding_margin']
axes[1, 1].hist(correct_margins, bins=35, alpha=0.70, label='Correct', color='#2a8c82')
axes[1, 1].hist(incorrect_margins, bins=35, alpha=0.70, label='Incorrect', color='#a65045')
axes[1, 1].axvline(0, color='#333333', linestyle=':')
axes[1, 1].set(
    title='True-prototype margin distribution',
    xlabel='Cosine similarity(true) − best wrong similarity',
    ylabel='Test images',
)
axes[1, 1].legend()

plt.tight_layout()
diagnostics_path = RESULTS_DIR / 'test_diagnostics.png'
plt.savefig(diagnostics_path, dpi=160, bbox_inches='tight')
plt.show()
display(confusion_pairs_df)
print('Saved per-class metrics:', per_class_path)
print('Saved diagnostics:', diagnostics_path)


## 8. Query-image species retrieval

`retrieve_image` embeds a JPG/PNG, ranks all 185 species prototypes and shows the nearest individual training references. Cosine similarity is a ranking score, not a calibrated probability. A low top score or small margin may indicate uncertainty, but open-set rejection requires a separately validated threshold.


In [ ]:
@torch.inference_mode()
def retrieve_image(image_path, top_k_species=5, top_k_images=5):
    image_path = Path(image_path)
    with Image.open(image_path) as image_file:
        query_image = image_file.convert('RGB')
    input_tensor = eval_transform(query_image).unsqueeze(0).to(device)

    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        query_embedding = model(input_tensor)
    query_embedding = F.normalize(query_embedding.float().cpu(), dim=1)

    species_scores = (query_embedding @ prototype_embeddings.T)[0]
    best_species_scores, best_species_indices = species_scores.topk(
        min(top_k_species, NUM_CLASSES)
    )
    species_result = pd.DataFrame({
        'species': [CLASS_NAMES[index] for index in best_species_indices.tolist()],
        'cosine_similarity': best_species_scores.tolist(),
    })

    image_scores = (query_embedding @ reference_embeddings.T)[0]
    best_image_scores, best_reference_positions = image_scores.topk(
        min(top_k_images, len(reference_embeddings))
    )
    nearest_rows = []
    for score, reference_position in zip(
        best_image_scores.tolist(), best_reference_positions.tolist()
    ):
        reference_row = reference_df.iloc[reference_position]
        nearest_rows.append({
            'species': reference_row['species'],
            'source': reference_row['source'],
            'image_path': reference_row['image_path'],
            'cosine_similarity': score,
        })
    nearest_images = pd.DataFrame(nearest_rows)

    display(query_image.resize((
        min(600, query_image.width),
        int(query_image.height * min(600, query_image.width) / query_image.width),
    )))
    display(species_result.style.format({'cosine_similarity': '{:.4f}'}))
    display(nearest_images.style.format({'cosine_similarity': '{:.4f}'}))

    figure, axes = plt.subplots(
        1, len(nearest_images), figsize=(3 * len(nearest_images), 3.4), squeeze=False
    )
    for axis, (_, nearest_row) in zip(axes.flat, nearest_images.iterrows()):
        with Image.open(DATASET_ROOT / nearest_row['image_path']) as reference_file:
            axis.imshow(reference_file.convert('RGB'))
        axis.set_title(
            f"{nearest_row['species']}\ncos={nearest_row['cosine_similarity']:.3f}",
            fontsize=8,
        )
        axis.axis('off')
    figure.suptitle('Nearest training reference images', y=1.02)
    plt.tight_layout()
    plt.show()
    return species_result, nearest_images

example_row = test_df.sample(1, random_state=SEED).iloc[0]
print('Expected species:', example_row['species'], '| source:', example_row['source'])
_ = retrieve_image(DATASET_ROOT / example_row['image_path'])


## 9. Completion checks and saved outputs

A successful run should produce these permanent Drive artifacts:

- `splits/leafsnap_group_split_seed42.csv` and its JSON fingerprint
- `checkpoints/triplet_resnet50/last.pt` for resuming
- `checkpoints/triplet_resnet50/best.pt` for final retrieval
- `results/triplet_resnet50/run_config.json`
- `results/triplet_resnet50/training_history.csv` and `training_curves.png`
- `results/triplet_resnet50/reference_gallery.csv` and `reference_dictionary.pt`
- `results/triplet_resnet50/test_embeddings.pt`
- `results/triplet_resnet50/test_metrics.json` and `domain_gap.json`
- `results/triplet_resnet50/test_predictions.csv`
- `results/triplet_resnet50/per_class_metrics.csv`
- `results/triplet_resnet50/top_confusion_pairs.csv` and `test_diagnostics.png`

All four requested model notebooks now use the same saved split. Compare their held-out results only after every notebook has completed on that identical split.
